In [ ]:
import pandas as pd

df = pd.read_csv("tripadvisor_attractions.csv")

print(df.shape)
print(df.dtypes)
print((df.isna().mean()*100).round(2))
df.head()

(5783, 11)
country            object
city               object
headline           object
price_raw          object
price_value       float64
currency           object
review_score      float64
review_count        int64
location_city      object
source_url         object
scraped_at_utc     object
dtype: object
country            0.00
city               0.00
headline           0.00
price_raw         43.33
price_value       43.33
currency          43.33
review_score       0.00
review_count       0.00
location_city      0.00
source_url         0.00
scraped_at_utc     0.00
dtype: float64


,country,city,headline,price_raw,price_value,currency,review_score,review_count,location_city,source_url,scraped_at_utc
0,UK,London,The London Pass®: 100+ Things To Do - Includes...,$135.67,135.67,$,3.7,863,London,https://www.tripadvisor.com/AttractionProductR...,2026-02-21T07:21:27.334134
1,UK,London,London Big Bus Hop-On Hop-Off Tour with Option...,$39.74,39.74,$,3.7,8071,London,https://www.tripadvisor.com/AttractionProductR...,2026-02-21T07:21:39.243625
2,UK,London,Tower of London and Crown Jewels Exhibition Ti...,$49.06,49.06,$,4.6,3387,London,https://www.tripadvisor.com/AttractionProductR...,2026-02-21T07:21:53.462772
3,UK,London,"Stonehenge, Windsor Castle, and Bath from London",NaN,NaN,NaN,4.8,9181,London,https://www.tripadvisor.com/AttractionProductR...,2026-02-21T07:22:07.293459
4,UK,Manchester,Alcotraz Prison Cocktail Experience in Manchester,$69.89,69.89,$,5.0,1203,Manchester,https://www.tripadvisor.com/AttractionProductR...,2026-02-21T07:22:45.384219


In [3]:
df["headline_clean"] = (
    df["headline"]
      .astype(str)
      .str.replace(r"\bclosed\b", "", case=False, regex=True)
      .str.replace(r"\btemporarily closed\b", "", case=False, regex=True)
      .str.strip()
)

In [4]:
dups = df.duplicated(subset=["headline","source_url"]).sum()
print("Duplicates:", dups)

df = df.drop_duplicates(subset=["headline","source_url"])

Duplicates: 3


In [5]:
df = df.drop_duplicates(subset=["headline", "source_url"])

In [6]:
df.duplicated(subset=["headline", "source_url"]).sum()

np.int64(0)

In [7]:
(df["city"].str.lower().str.strip() == df["location_city"].str.lower().str.strip()).mean()

np.float64(1.0)

In [8]:
df = df.drop(columns=["location_city"])

In [9]:
df

,country,city,headline,price_raw,price_value,currency,review_score,review_count,source_url,scraped_at_utc,headline_clean
0,UK,London,The London Pass®: 100+ Things To Do - Includes...,$135.67,135.67,$,3.7,863,https://www.tripadvisor.com/AttractionProductR...,2026-02-21T07:21:27.334134,The London Pass®: 100+ Things To Do - Includes...
1,UK,London,London Big Bus Hop-On Hop-Off Tour with Option...,$39.74,39.74,$,3.7,8071,https://www.tripadvisor.com/AttractionProductR...,2026-02-21T07:21:39.243625,London Big Bus Hop-On Hop-Off Tour with Option...
2,UK,London,Tower of London and Crown Jewels Exhibition Ti...,$49.06,49.06,$,4.6,3387,https://www.tripadvisor.com/AttractionProductR...,2026-02-21T07:21:53.462772,Tower of London and Crown Jewels Exhibition Ti...
3,UK,London,"Stonehenge, Windsor Castle, and Bath from London",NaN,NaN,NaN,4.8,9181,https://www.tripadvisor.com/AttractionProductR...,2026-02-21T07:22:07.293459,"Stonehenge, Windsor Castle, and Bath from London"
4,UK,Manchester,Alcotraz Prison Cocktail Experience in Manchester,$69.89,69.89,$,5.0,1203,https://www.tripadvisor.com/AttractionProductR...,2026-02-21T07:22:45.384219,Alcotraz Prison Cocktail Experience in Manchester
...,...,...,...,...,...,...,...,...,...,...,...
5778,Indonesia,Surabaya,Bukit Kapur Jaddih,NaN,NaN,NaN,3.8,95,https://www.tripadvisor.com/Attraction_Review-...,2026-02-22T05:41:24.074082,Bukit Kapur Jaddih
5779,Indonesia,Surabaya,Trowulan Tour Ruin of Majapahit Kingdom from S...,$85.00,85.00,$,5.0,17,https://www.tripadvisor.com/AttractionProductR...,2026-02-22T05:41:36.968216,Trowulan Tour Ruin of Majapahit Kingdom from S...
5780,Indonesia,Surabaya,Mount Bromo Sunrise Shared Guided Tour from Su...,$70.00,70.00,$,4.9,8,https://www.tripadvisor.com/AttractionProductR...,2026-02-22T05:41:48.343494,Mount Bromo Sunrise Shared Guided Tour from Su...
5781,Indonesia,Surabaya,Batu City Square,NaN,NaN,NaN,4.0,540,https://www.tripadvisor.com/Attraction_Review-...,2026-02-22T05:41:59.490574,Batu City Square


In [11]:
df["price_value"].isna().sum()

np.int64(2506)

In [12]:
import pandas as pd

df["has_price"] = df["price_value"].notna()

city_missing = (
    df.groupby(["country", "city"])
      .agg(
          total_rows=("source_url", "count"),
          priced_rows=("has_price", "sum"),
          missing_price_rows=("has_price", lambda s: (~s).sum()),
          missing_price_pct=("has_price", lambda s: (1 - s.mean()) * 100),
      )
      .reset_index()
)

# Qiyməti tam çəkilməyən şəhərlər yuxarıda olsun
city_missing = city_missing.sort_values("missing_price_pct", ascending=False)

city_missing.head(30)

,country,city,total_rows,priced_rows,missing_price_rows,missing_price_pct
14,Denmark,Billund,62,10,52,83.870968
29,India,Delhi,32,8,24,75.000000
26,Iceland,Egilsstadir,69,20,49,71.014493
41,Malta,St. Julian's,31,9,22,70.967742
57,Spain,Madrid,3,1,2,66.666667
13,Denmark,Aarhus,79,37,42,53.164557
32,Indonesia,Jakarta,119,57,62,52.100840
33,Indonesia,Surabaya,122,59,63,51.639344
49,Netherlands,Amsterdam,84,41,43,51.190476
54,South Korea,Jeju,2,1,1,50.000000


In [13]:
df[df["price_value"].isna()][["country","city","headline","source_url"]] \
  .to_csv("urls_missing_price.csv", index=False, encoding="utf-8-sig")

In [1]:
import pandas as pd

# 1) Faylları oxu
base = pd.read_csv("urls_missing_price.csv")
patch = pd.read_csv("missing_price_rescraped.csv")

# 2) source_url ilə LEFT JOIN (base əsasdır)
df_merged = base.merge(patch, on="source_url", how="left")

# 3) Sütunları daha səliqəli et (istəsən)
# (əsas price sütunu yarat)
df_merged["price_value"] = df_merged["price_value_new"]
df_merged["price_raw"]   = df_merged["price_raw_new"]
df_merged["currency"]    = df_merged["currency_new"]

# 4) Artıq lazımsız köməkçi sütunları sil
df_merged = df_merged.drop(columns=[
    "price_value_new",
    "price_raw_new",
    "currency_new",
    "rescraped_at_utc"
], errors="ignore")

# 5) Nəticəni saxla
df_merged.to_csv("missing_price_merged_full.csv", index=False, encoding="utf-8-sig")

print("✅ Hazır fayl: missing_price_merged_full.csv")
print("Rows:", df_merged.shape)
print("Missing price:", df_merged["price_value"].isna().sum())

✅ Hazır fayl: missing_price_merged_full.csv
Rows: (2506, 7)
Missing price: 31


In [2]:
import pandas as pd

base = pd.read_csv("tripadvisor_attractions.csv")
patch = pd.read_csv("missing_price_merged_full.csv")

df = base.merge(
    patch[["source_url", "price_value", "price_raw", "currency"]],
    on="source_url",
    how="left",
    suffixes=("", "_new")
)

In [3]:
mask = df["price_value"].isna() & df["price_value_new"].notna()

df.loc[mask, "price_value"] = df.loc[mask, "price_value_new"]
df.loc[mask, "price_raw"]   = df.loc[mask, "price_raw_new"]
df.loc[mask, "currency"]    = df.loc[mask, "currency_new"]

In [5]:
df_clean = df.dropna(subset=["price_value"])

In [7]:
df["price_value"].isna().sum()

np.int64(31)

In [14]:
df[df["price_value"].isna()][
    ["headline", "city", "country", "source_url"]
].head(35)

,headline,city,country,source_url
773,Basílica de Nuestra Señora de Zapopan,Guadalajara,Mexico,https://www.tripadvisor.com/Attraction_Review-...
824,"A Journey to Christmas in Moscow, Russia",Sydney,Australia,https://www.tripadvisor.com/AttractionProductR...
2361,BESycle E-bike,Sliema,Malta,https://www.tripadvisor.com/Attraction_Review-...
2413,Iran Mall,St. Julian's,Malta,https://www.tripadvisor.com/Attraction_Review-...
2651,Horseback riding on the beach,Akureyri,Iceland,https://www.tripadvisor.com/AttractionProductR...
2724,Akureyrarkirkja,Akureyri,Iceland,https://www.tripadvisor.com/Attraction_Review-...
2732,Icelandic Wartime Museum,Egilsstadir,Iceland,https://www.tripadvisor.com/Attraction_Review-...
2782,Seydisfjordur Church,Egilsstadir,Iceland,https://www.tripadvisor.com/Attraction_Review-...
2905,Truc Lam An Tam Zen Monastery,Hanoi,Vietnam,https://www.tripadvisor.com/Attraction_Review-...
2911,Amie Travel| Customized Your Dream Holiday,Hanoi,Vietnam,https://www.tripadvisor.com/Attraction_Review-...


In [13]:
df.groupby(["country", "city"])["price_value"].apply(lambda x: x.isna().sum()).reset_index(name="missing_price_count")

,country,city,missing_price_count
0,Australia,Brisbane,0
1,Australia,Melbourne,0
2,Australia,Sydney,1
3,Austria,Innsbruck,0
4,Austria,Salzburg,0
...,...,...,...
74,USA,Miami,0
75,USA,New York,0
76,Vietnam,Da Nang,0
77,Vietnam,Hanoi,2


In [18]:
df = df.drop(columns=["price_raw"])

In [19]:
df.to_csv("tripadvisor_attractions_final.csv", index=False, encoding="utf-8-sig")

In [ ]:
df[]